In [1]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 15.8 MB/s eta 0:00:00


In [2]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded.")

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded.


In [4]:
from qwen_vl_utils import process_vision_info

def call_vlm(prompt: str, images: list = None) -> str:
    """
    prompt: text instruction/question
    images: list of local image file paths (optional)
    Returns: model's text response as a string
    """
    content = []
    if images:
        for img_path in images:
            content.append({"type": "image", "image": img_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_special_tokens=False
    )
    return output_text[0]

In [6]:
# Text-only test
print(call_vlm("What is 2+2?"))

# With an image (upload one via the folder icon on the left sidebar first)
# print(call_vlm("What's in this image?", images=["/content/your_test_image.jpg"]))

2+2 is equal to 4.


In [8]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [9]:
from PIL import Image

img_path = "/content/Gemini_Generated_Image_3cq4s23cq4s23cq4.png"
img = Image.open(img_path)
img = img.resize((512, 512))
img.save("/content/resized_test.png")

In [14]:
print(call_vlm("What's in this image?", images=["/content/resized_test.png"]))

The image is a collage that showcases two different applications and their respective use cases. 

1. **The Solo Burden**:
   - This application appears to be designed for caregivers, particularly mothers, who are managing the care of a baby and handling household chores.
   - The interface includes reminders for medication and blood pressure checks, indicating its focus on health and wellness management.
   - The application seems to be designed to help caregivers manage their tasks efficiently and stay organized.

2. **The Distributed Future with Amara**:
   - This application is presented as a live working prototype, suggesting it is a new or experimental app.
   - It shows a family setting where multiple family members are interacting with the app, indicating its use in a household context.
   - The app appears to be designed for managing tasks and coordinating activities among family members, promoting a distributed approach to household management.

The image highlights the poten

In [12]:
from PIL import Image
from qwen_vl_utils import process_vision_info

def call_vlm(prompt: str, images: list = None) -> str:
    content = []
    if images:
        for img_path in images:
            img = Image.open(img_path)
            img.thumbnail((512, 512))  # resize in-place, preserves aspect ratio, prevents OOM
            resized_path = img_path.rsplit(".", 1)[0] + "_resized.png"
            img.save(resized_path)
            content.append({"type": "image", "image": resized_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_special_tokens=False)
    return output_text[0]